# Torus breakdown between the Neimark–Sacker line and chaos?

Open question 4 of `article_chaos/report_v2.txt`: *is the route period doubling → Neimark–Sacker → torus breakdown? Poincaré sections in the < 1 MHz window
above the Neimark–Sacker line would show the torus.* The maps (`lyapunov_heatmap*`) have a grid step of ≈ 1.1 MHz in $g_2$, so at $\varepsilon_d/2g_2 = 5.06$
the whole window between the Neimark–Sacker point (76.7 MHz) and the first chaotic grid point (77.6 MHz) has never been sampled.

Here the attractor born from the satellite is followed by **parameter continuation** in $g_2$ (each point starts from the end state of the previous one),
with steps down to 0.2 kHz, and characterised by
- the stroboscopic (Poincaré) section $a(nT_a)$ or $a(2nT_a)$: a point for a periodic orbit, a closed curve for a torus, a cloud for chaos;
- the Lyapunov spectrum and the convergence of $\lambda_1$ with the averaging time ($\lambda_1 \to 0$ as $1/T$ on a torus);
- the rotation number on the section, compared with the angle of the Hill multipliers at the Neimark–Sacker point;
- escape times from the torus and lifetimes of the chaotic transients around the point where the torus disappears.

**Conventions.** $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$ (static pump), $\varepsilon_d = 10.12\,g_2$ unless stated otherwise, $\kappa_b = \pi/10.4$ in the code
convention ($-2\kappa_b\dot\phi_b$); **the article's $\kappa_b$ is twice this one**, so $\lambda/\kappa_b$ values printed here are $2\times$ the article's and $\kappa_b T$ values are half
the article's. Times are mostly given in memory periods $T_a = 2\pi/\omega_a \approx 0.248$ ns. The drive period is $T_a/2$; the period-2 orbit has period $2T_a$, so its
section at $2T_a$ is a single point and the torus born from it is a single closed curve. Flux form (`SystemATS`, `equation_1ODE!`) integrated with `Vern9`, tolerance $10^{-10}$.

**Result.** A torus does appear after the Neimark–Sacker point, but it does not break down gradually. Along $\varepsilon_d = 10.12\,g_2$ the route is:
satellite → period doubling (75.25 MHz) → Neimark–Sacker (76.70 MHz), torus → **torus doubling** (≈ 77.0 MHz) → **abrupt** jump at $g_c = 77.249$ MHz to a large
chaotic attractor. At $g_c$ the doubled torus is still smooth and transversally stable, there is no visible phase locking, and the chaotic set already exists below $g_c$ as a
chaotic saddle (long transients). The whole quasi-periodic window is ≈ 0.55 MHz wide, between two grid points of the Lyapunov maps, and has the same width (0.55–0.7 MHz) at
$\varepsilon_d/2g_2 = 3, 4, 6.5$. Details in the conclusion at the end.

In [ ]:
using ChaosTools, JSON, Statistics, LinearAlgebra, Base.Threads
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")
println("threads: ", nthreads())

In [ ]:
ϕ_quantum = 1/2
ω_a = 25.338776456203686; ω_b = 2ω_a
φ_a = 0.11;  Z_a = 2ϕ_quantum^2*φ_a^2; E_Ca = Z_a*ω_a/2
φ_b = 0.204; Z_b = 2ϕ_quantum^2*φ_b^2; E_Cb = Z_b*ω_b/2
E_J = 37.12*2π*1.5
ω_d = ω_b; ω_p = 2ω_a - ω_d          # static pump (ω_p = 0)
κ_b = π/10.4; κ_a = 0.0              # code convention (−2κ_b φ̇_b): the article's κ_b is 2× this
T_a = 2π/ω_a                         # memory period; the drive period is T_a/2
ratio_ref = 10.12                    # ε_d/g2 (ε_d/2g2 = 5.06, working point of the article)

g2_of(ϵ_p) = sin(ϵ_p)*E_J*φ_a^2*φ_b
ϵ_p_of(g) = asin(g*2π*1e-3/(E_J*φ_a^2*φ_b))      # g = g2/2π in MHz

out_dir = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/"
fig_dir = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/"

**Helpers.** `strobe` integrates from a state at $t = 0$ and returns the states at $t = nT_a$. Every $T_a$ is a multiple of the drive period, so the last state
seeds the next run, again started at $t = 0$ (as in `lyapunov_heatmap_satellite_start`). The state of `equation_1ODE!` is
$u = (\sqrt2\,\mathrm{Re}\,a,\ \sqrt2\,\mathrm{Re}\,b,\ \sqrt2\,\omega_a\mathrm{Im}\,a,\ \sqrt2\,\omega_b\mathrm{Im}\,b)$.
The rotation number is the mean angle per iterate of the section around its centre, in turns.

In [ ]:
system_0 = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, 0.0, 0.0, 0.0)
sp = prepare(system_0)
ep = prepare(system_0, MeanField(κ_a, κ_b))
params(g; r = ratio_ref) = (ϵ_p = ϵ_p_of(g); SimulationParams(sp, prepare(DriveTwoModesBasics(r*g2_of(ϵ_p), ω_d, 0.0, ϵ_p, ω_p, 0.0)), ep))

amp_a(u) = (u[1] + 1im*u[3]/ω_a)/sqrt(2)
amp_b(u) = (u[2] + 1im*u[4]/ω_b)/sqrt(2)
u_from(a, b) = [sqrt(2)*real(a), sqrt(2)*real(b), sqrt(2)*ω_a*imag(a), sqrt(2)*ω_b*imag(b)]

const TOL = 1e-10
# states at t = n T_a for n = n_tr, n_tr + step, …, n_tr + n_rec
function strobe(u0, g; r = ratio_ref, n_tr = 0, n_rec = 2000, step = 2)
    prob = ODEProblem(equation_1ODE!, copy(u0), (0.0, (n_tr + n_rec)*T_a), params(g; r = r))
    solve(prob, Vern9(); abstol = TOL, reltol = TOL, saveat = (n_tr:step:n_tr + n_rec)*T_a, maxiters = 10^9).u
end
section(U) = [amp_a(u) for u in U]

dyn(u0, g; r = ratio_ref) = CoupledODEs(equation_1ODE!, copy(u0), params(g; r = r); diffeq = (alg = Vern9(), abstol = TOL, reltol = TOL))
# largest exponent and spectrum, in units of κ_b (code convention); u0 is already on the attractor
λ_max(u0, g; N = 2000, r = ratio_ref) = lyapunov(dyn(u0, g; r = r), N*T_a; Δt = T_a, d0 = 1e-8, d0_upper = 1e-5, d0_lower = 1e-11)/κ_b
λ_spec(u0, g; N = 4000, k = 4) = lyapunovspectrum(dyn(u0, g), N, k; Δt = T_a) ./ κ_b

# smallest p with A[n+p] = A[n] (relative tolerance), 0 if none up to pmax
function period_of(A; tol = 1e-6, pmax = 200)
    s = mean(abs.(A))
    for p in 1:pmax
        maximum(abs.(A[1+p:end] .- A[1:end-p])) < tol*s && return p
    end
    return 0
end
rotation(A) = (z = A .- mean(A); mean(mod.(diff(angle.(z)) .+ π, 2π) .- π)/2π)

## 0. Hill reference

Floquet multipliers of the period-2 orbit from `Hill_method_doubling_corrected.ipynb` (`floquet_after_period_doubling_12_10_2026.json`, per period $2T_a$)
and of the satellite from `Hill_method_corrected.ipynb` (`floquet_before_period_doubling_12_10_2026.json`). The Neimark–Sacker point is where $\max|\mu|$ crosses 1;
the angle $\arg\mu/2\pi$ of the critical pair there is the rotation number expected on the newborn torus (section at $2T_a$).

In [ ]:
function load_hill(name)
    h = JSON.parsefile(out_dir*name)
    μ = [Float64.(R) .+ 1im .* Float64.(I) for (R, I) in zip(h["multipliers_real"], h["multipliers_imag"])]
    g = Float64.(h["g2_MHz"]); p = sortperm(g)
    return g[p], μ[p]
end
# first crossing of max|μ| = 1 going up in g2, linearly interpolated; returns (g, index)
function unit_crossing(g, μ)
    m = [maximum(abs.(x)) for x in μ]
    k = findfirst(i -> m[i] < 1 && m[i+1] > 1, 1:length(m)-1)
    return g[k] + (1 - m[k])*(g[k+1] - g[k])/(m[k+1] - m[k]), k
end
g_hill_1, μ_hill_1 = load_hill("floquet_before_period_doubling_12_10_2026.json")
g_hill_2, μ_hill_2 = load_hill("floquet_after_period_doubling_12_10_2026.json")
g_PD_hill, _ = unit_crossing(g_hill_1, μ_hill_1)
g_NS_hill, k_NS = unit_crossing(g_hill_2, μ_hill_2)
# angle of the leading multiplier pair of the period-2 orbit (turns per 2T_a)
θ_hill = [abs(angle(x[argmax(abs.(x))]))/2π for x in μ_hill_2]
θ_NS_hill = θ_hill[k_NS] + (θ_hill[k_NS+1] - θ_hill[k_NS])*(g_NS_hill - g_hill_2[k_NS])/(g_hill_2[k_NS+1] - g_hill_2[k_NS])
println("Hill: period doubling at g2/2π = ", round(g_PD_hill, digits=2), " MHz, Neimark–Sacker at ", round(g_NS_hill, digits=2),
        " MHz, arg μ/2π = ", round(θ_NS_hill, digits=4), " per 2T_a")

## 1. Continuation from the satellite, 74.9 → 79 MHz

Seed: from $a(0) = 4.2i$, $b(0) = 0$ at $g_2/2\pi = 74.9$ MHz the trajectory settles on a large-amplitude orbit of period $T_a$: the satellite (the cat has $|a| \approx 2.2$ and
$\langle|b|\rangle \approx 0.1$). It is then followed up in steps of 20 kHz, with $3000\,T_a$ ($\approx 225/\kappa_b$) between steps. The diagram shows $\mathrm{Re}\,a(nT_a)$ over
the last $600\,T_a$: one branch for the satellite, two after period doubling.

In [ ]:
g_seed = 74.9
u_seed = strobe(u_from(4.2im, 0.0), g_seed; n_rec = 4000)[end]
U = strobe(u_seed, g_seed; n_rec = 200, step = 1)
println("seed: period ", period_of(section(U)), " T_a, |a| = ", round(mean(abs.(section(U))), digits=3),
        ", <|b|> = ", round(mean(abs.(amp_b.(U))), digits=3))

g_coarse = collect(74.9:0.02:79.0)
states_coarse = Vector{Vector{Float64}}(undef, length(g_coarse))
let u = u_seed
    for (i, g) in enumerate(g_coarse)
        u = strobe(u, g; n_rec = 3000)[end]
        states_coarse[i] = u
    end
end
A_coarse = Vector{Vector{ComplexF64}}(undef, length(g_coarse))
λ_coarse = zeros(length(g_coarse))
@threads for i in eachindex(g_coarse)
    A_coarse[i] = section(strobe(states_coarse[i], g_coarse[i]; n_rec = 600, step = 1))
    λ_coarse[i] = λ_max(states_coarse[i], g_coarse[i]; N = 2000)
end
per_coarse = period_of.(A_coarse)
i_PD = findfirst(==(2), per_coarse); i_QP = findfirst(i -> per_coarse[i] == 0 && i > i_PD, eachindex(per_coarse))
i_ch = findfirst(>(0.3), λ_coarse)
println("simulation: period 2 from ", g_coarse[i_PD], " MHz, aperiodic from ", g_coarse[i_QP], " MHz, λ1 > 0.3 κ_b from ", g_coarse[i_ch], " MHz")

In [ ]:
xs = vcat([fill(g_coarse[i], length(A_coarse[i])) for i in eachindex(g_coarse)]...)
ys = vcat([real.(A) for A in A_coarse]...)
p_bif = scatter(xs, ys, ms = 0.6, msw = 0, color = :black, label = "", ylabel = L"\mathrm{Re}\,a(nT_a)")
p_λ = plot(g_coarse, λ_coarse, color = Perso_palette_hex[3], lw = 1.5, label = "", ylabel = L"\lambda_1/\kappa_b", xlabel = L"g_2/2\pi"*" (MHz)")
hline!(p_λ, [0], color = :gray, ls = :dot, label = "")
for p in (p_bif, p_λ)
    vline!(p, [g_PD_hill], color = Perso_palette_hex[5], ls = :dash, lw = 1, label = (p === p_bif ? "Hill: period doubling" : ""))
    vline!(p, [g_NS_hill], color = Perso_palette_hex[2], ls = :dash, lw = 1, label = (p === p_bif ? "Hill: Neimark–Sacker" : ""))
    plot!(p, xlims = (g_coarse[1], g_coarse[end]), grid = false)
end
plot!(p_bif, legend = :bottomleft)
p_coarse = plot(p_bif, p_λ, layout = grid(2, 1, heights = [0.65, 0.35]), link = :x, size = (2*size_px_car[1], 1.6*size_px_car[2]), left_margin = 5mm, bottom_margin = 3mm)
savefig(p_coarse, fig_dir*"torus_bifurcation_diagram_19_10_2026.png")
display("image/png", p_coarse)   # PNG only: IJulia would also store an SVG with every point

## 2. Zoom on the window: Lyapunov spectrum and rotation number

Continuation from 76.50 MHz in steps of 10 kHz ($6000\,T_a$ between steps, the relaxation is slow near the bifurcations), then 4 kHz up to 77.248 MHz.
At each point: section at $2T_a$ over $8000\,T_a$, the four Lyapunov exponents over $4000\,T_a$, and the rotation number.

In [ ]:
g_zoom = vcat(collect(76.50:0.01:77.5))#, [77.244, 77.248])
states_zoom = Vector{Vector{Float64}}(undef, length(g_zoom))
let u = states_coarse[argmin(abs.(g_coarse .- 76.50))]
    for (i, g) in enumerate(g_zoom)
        u = strobe(u, g; n_rec = 6000)[end]
        states_zoom[i] = u
    end
end
A_zoom = Vector{Vector{ComplexF64}}(undef, length(g_zoom))
λ_zoom = zeros(length(g_zoom), 4)
@threads for i in eachindex(g_zoom)
    A_zoom[i] = section(strobe(states_zoom[i], g_zoom[i]; n_rec = 8000, step = 2))
    λ_zoom[i, :] = λ_spec(states_zoom[i], g_zoom[i]; N = 4000)
end
rot_zoom = rotation.(A_zoom)
per_zoom = period_of.(A_zoom)
amin_zoom = [minimum(abs.(A)) for A in A_zoom]; amax_zoom = [maximum(abs.(A)) for A in A_zoom]
for i in eachindex(g_zoom)
    i % 4 == 1 || i > length(g_zoom) - 3 || continue
    println(rpad(g_zoom[i], 7), "  period(2T_a) ", lpad(per_zoom[i], 3), "  λ/κ_b ", round.(λ_zoom[i, :], digits=3),
            "  rot ", round(rot_zoom[i], digits=4), "  |a| ", round(amin_zoom[i], digits=4), "–", round(amax_zoom[i], digits=4))
end
#amax_zoom[end] - amin_zoom[end] < 0.5 || error("the continuation left the torus before 77.248 MHz")
# Neimark–Sacker: zero crossing of λ1 (= λ2, the complex pair) from below; near it the spiral relaxes too slowly for the period test
i_T2 = findfirst(>(0), λ_zoom[:, 1])
g_NS_sim = g_zoom[i_T2-1] - λ_zoom[i_T2-1, 1]*(g_zoom[i_T2] - g_zoom[i_T2-1])/(λ_zoom[i_T2, 1] - λ_zoom[i_T2-1, 1])
# torus doubling: λ2 reaches 0 between the NS point and 77.1 MHz
w = findall(g -> g_NS_sim + 0.1 < g < 77.1, g_zoom)
g_TD_sim = g_zoom[w[argmax(λ_zoom[w, 2])]]
println("simulation: Neimark–Sacker at ≈ ", round(g_NS_sim, digits=3), " MHz (Hill ", round(g_NS_hill, digits=2), "), torus doubling (max of λ2) at ≈ ", g_TD_sim, " MHz")

In [ ]:
lab = [L"\lambda_1", L"\lambda_2", L"\lambda_3", L"\lambda_4"]; col = Perso_palette_hex[[3, 4, 5, 6]]
p_spec = plot(ylabel = L"\lambda_i/\kappa_b", legend = :bottomleft, grid = false)
for k in 1:4
    plot!(p_spec, g_zoom, λ_zoom[:, k], color = col[k], lw = 1.5, marker = :circle, ms = 1.5, msw = 0, label = lab[k])
end
hline!(p_spec, [0], color = :gray, ls = :dot, label = "")
p_rot = plot(g_zoom[i_T2:end], rot_zoom[i_T2:end], color = Perso_palette_hex[3], lw = 1.5, label = "simulation",
             ylabel = "rotation number", xlabel = L"g_2/2\pi"*" (MHz)", legend = :topleft, grid = false)
w = findall(g -> g_zoom[1] <= g <= g_zoom[end], g_hill_2)
plot!(p_rot, g_hill_2[w], θ_hill[w], color = Perso_palette_hex[2], ls = :dash, marker = :diamond, ms = 3, msw = 0, label = "Hill: arg μ/2π, leading pair")
scatter!(p_rot, [g_NS_hill], [θ_NS_hill], marker = :star5, ms = 7, msw = 0, color = Perso_palette_hex[5], label = "Hill at the Neimark–Sacker point")
for p in (p_spec, p_rot)
    vline!(p, [g_NS_hill], color = Perso_palette_hex[2], ls = :dash, lw = 1, label = "")
    vline!(p, [g_TD_sim], color = :gray, ls = :dashdot, lw = 1, label = "")
    plot!(p, xlims = (g_zoom[1], g_zoom[end] + 0.01))
end
p_zoom = plot(p_spec, p_rot, layout = (2, 1), link = :x, size = (2*size_px_car[1], 1.8*size_px_car[2]), left_margin = 5mm, bottom_margin = 3mm)
savefig(p_zoom, fig_dir*"torus_lyapunov_rotation_19_10_2026.pdf")
display("image/png", p_zoom)   # PNG only: IJulia would also store an SVG with every point

In [ ]:
extrema(sum(λ_zoom, dims=2))


Before the Neimark–Sacker point, $\lambda_1 = \lambda_2$ (one complex pair) rises to 0, and the rotation number of the decaying spiral matches the Hill angle.
After it, $\lambda_1 \approx 0$ (the slightly positive values are the $1/T$ bias, see the next cell) and the period-2 orbit is replaced by an invariant curve.
The dash-dotted line marks where $\lambda_2$ touches 0 a second time: the torus doubles (section 3).

### Convergence of $\lambda_1$ with the averaging time

On a torus the separation grows polynomially, so the finite-time estimate behaves as $\lambda_1 \approx c/T$; on a periodic orbit it converges to a negative value, in chaos to a positive one.

In [ ]:
g_conv = [76.60, 76.80, 77.00, 77.20]
N_conv = [500, 2000, 8000, 32000]
λ_conv = zeros(length(g_conv), length(N_conv))
@threads for k in 1:length(g_conv)*length(N_conv)
    i, j = divrem(k - 1, length(N_conv)) .+ 1
    λ_conv[i, j] = λ_max(states_zoom[argmin(abs.(g_zoom .- g_conv[i]))], g_conv[i]; N = N_conv[j])
end
κT = N_conv .* T_a .* κ_b
for i in eachindex(g_conv)
    println(g_conv[i], " MHz:  λ1/κ_b = ", round.(λ_conv[i, :], digits=5), "   λ1 κ_b T = ", round.(λ_conv[i, :] .* κT, digits=2))
end
p_conv = plot(xscale = :log10, yscale = :log10, xlabel = L"\kappa_b T", ylabel = L"|\lambda_1|/\kappa_b", legend = :bottomleft, grid = false,
              size = (1.5*size_px_car[1], size_px_car[2]), left_margin = 5mm, bottom_margin = 5mm)
for i in eachindex(g_conv)
    plot!(p_conv, κT, abs.(λ_conv[i, :]), marker = (λ_conv[i, end] < 0 ? :utriangle : :circle), ms = 4, msw = 0,
          color = Perso_palette_hex[[5, 3, 2, 6]][i], label = string(g_conv[i], " MHz", λ_conv[i, end] < 0 ? " (negative)" : ""))
end
plot!(p_conv, κT, 3 ./ κT, color = :gray, ls = :dash, label = L"\propto 1/T")
display("image/png", p_conv)   # PNG only: IJulia would also store an SVG with every point

## 3. Poincaré sections

Section at $2T_a$ in the memory plane. Each panel has its own scale (the torus is $\sim 0.1$ wide around $|a| \approx 5.2$).
The last two panels: even and odd iterates at 77.10 MHz (both fill both loops, so it is **one** curve winding twice — a torus doubling, not a pair of tori
exchanged by the map), and the chaotic attractor at 77.26 MHz with the torus at 77.248 MHz.

In [ ]:
g_show = [76.70, 76.76, 76.90, 77.00, 77.06, 77.20, 77.248]
u_248 = states_zoom[end]
A_chaos = section(strobe(u_248, 77.26; n_tr = 4000, n_rec = 16000, step = 2))
panels = []
for g in g_show
    A = A_zoom[argmin(abs.(g_zoom .- g))]
    push!(panels, scatter(real.(A), imag.(A), ms = 0.8, msw = 0, color = :black, label = "", title = string(g, " MHz"), grid = false))
end
A = A_zoom[argmin(abs.(g_zoom .- 77.10))]
p_eo = scatter(real.(A[1:2:end]), imag.(A[1:2:end]), ms = 0.8, msw = 0, color = Perso_palette_hex[3], label = "even", grid = false, title = "77.1 MHz, even/odd")
scatter!(p_eo, real.(A[2:2:end]), imag.(A[2:2:end]), ms = 0.8, msw = 0, color = Perso_palette_hex[5], label = "odd", legend = :bottomleft)
p_ch = scatter(real.(A_chaos), imag.(A_chaos), ms = 0.6, msw = 0, color = Perso_palette_hex[4], label = "77.26 MHz (chaos)", grid = false, title = "chaos vs torus")
scatter!(p_ch, real.(A_zoom[end]), imag.(A_zoom[end]), ms = 0.6, msw = 0, color = :black, label = "77.248 MHz (torus)", legend = :topleft)
p_sec = plot(panels..., p_eo, p_ch, layout = (3, 3), size = (3*size_px_car[1], 3*size_px_car[2]), xlabel = L"\mathrm{Re}\,a", ylabel = L"\mathrm{Im}\,a",
             left_margin = 4mm, bottom_margin = 3mm, titlefontsize = 8)
savefig(p_sec, fig_dir*"torus_poincare_sections_19_10_2026.png")
display("image/png", p_sec)   # PNG only: IJulia would also store an SVG with every point

## 4. End of the torus

**Threshold.** From the doubled torus at 77.248 MHz, $g_2$ is raised in steps of 0.2 kHz, $40\,000\,T_a$ ($\approx 3000/\kappa_b$) per step. The trajectory has left the torus when
$|a(2nT_a)|$ leaves the band of the torus widened by 0.05.

In [ ]:
band = (minimum(abs.(A_zoom[end])) - 0.05, maximum(abs.(A_zoom[end])) + 0.05)
escape_index(A) = findfirst(x -> !(band[1] < abs(x) < band[2]), A)
g_thr = collect(77.2482:0.0002:77.2510)
esc_thr = fill(NaN, length(g_thr))
let u = u_248
    for (i, g) in enumerate(g_thr)
        U = strobe(u, g; n_rec = 40000, step = 2)
        k = escape_index(section(U))
        if isnothing(k)
            u = U[end]
        else
            esc_thr[i] = 2(k - 1)
            break
        end
    end
end
i_c = findfirst(!isnan, esc_thr)
g_c = (g_thr[i_c-1] + g_thr[i_c])/2
println("torus kept up to ", g_thr[i_c-1], " MHz (40000 T_a), left after ", esc_thr[i_c], " T_a at ", g_thr[i_c], " MHz  →  g_c ≈ ", g_c, " MHz")
u_c = strobe(u_248, g_thr[i_c-1]; n_rec = 4000)[end]      # on the torus just below g_c
println("spectrum on the torus at ", g_thr[i_c-1], " MHz: λ/κ_b = ", round.(λ_spec(u_c, g_thr[i_c-1]; N = 8000), digits=4))

**Above $g_c$: escape time from the ghost of the torus.** Eight starting points spread along the torus just below $g_c$, run at $g_c + \delta$.
A power law $\tau \propto \delta^{-\gamma}$ is expected for a crisis ($\gamma \ge 1/2$ in two-dimensional maps) or a saddle-node bottleneck ($\gamma = 1/2$).
The right panel shows where the trajectories leave the torus ($\delta = 1$ kHz: the 30 iterates before escape).

In [ ]:
U0 = strobe(u_c, g_thr[i_c-1]; n_rec = 400, step = 2)
starts = U0[1:25:end][1:8]
δ_list = [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]
τ_esc = fill(NaN, length(δ_list), length(starts))
tails = Vector{Vector{ComplexF64}}(undef, length(starts))
@threads for k in 1:length(δ_list)*length(starts)
    i, j = divrem(k - 1, length(starts)) .+ 1
    A = section(strobe(starts[j], g_c + δ_list[i]; n_rec = 30000, step = 2))
    e = escape_index(A)
    τ_esc[i, j] = isnothing(e) ? NaN : 2(e - 1)
    δ_list[i] == 0.001 && !isnothing(e) && (tails[j] = A[max(1, e-30):e])
end
τ_med = [median(filter(!isnan, τ_esc[i, :])) for i in eachindex(δ_list)]
γ_fit = -([ones(length(δ_list)) log.(δ_list)] \ log.(τ_med))[2]
println("median escape time (T_a): ", Dict(zip(δ_list, τ_med)), "\nγ ≈ ", round(γ_fit, digits=2), " (direct fit, no offset)")

p_esc = scatter(repeat(δ_list, 1, length(starts))[:]*1e3, τ_esc[:], xscale = :log10, yscale = :log10, ms = 3, msw = 0, color = Perso_palette_hex[4], label = "single runs",
                xlabel = L"(g_2 - g_c)/2\pi"*" (kHz)", ylabel = L"\tau/T_a", grid = false)
plot!(p_esc, δ_list*1e3, τ_med, color = Perso_palette_hex[3], lw = 1.5, marker = :circle, ms = 4, msw = 0, label = "median")
plot!(p_esc, δ_list*1e3, τ_med[2]*(δ_list/δ_list[2]).^(-0.5), color = :gray, ls = :dash, label = L"\propto \delta^{-1/2}")
p_loc = scatter(real.(A_zoom[end]), imag.(A_zoom[end]), ms = 0.6, msw = 0, color = :black, label = "torus, 77.248 MHz", grid = false,
                xlabel = L"\mathrm{Re}\,a", ylabel = L"\mathrm{Im}\,a")
for j in eachindex(tails)
    isassigned(tails, j) && plot!(p_loc, real.(tails[j]), imag.(tails[j]), color = Perso_palette_hex[5], lw = 0.8, marker = :circle, ms = 1.5, msw = 0, label = (j == 1 ? "before escape" : ""))
end
xl = extrema(real.(A_zoom[end])); yl = extrema(imag.(A_zoom[end]))
plot!(p_loc, xlims = (xl[1] - 0.05, xl[2] + 0.05), ylims = (yl[1] - 0.05, yl[2] + 0.05), legend = :topleft)
p_end = plot(p_esc, p_loc, layout = (1, 2), size = (2.4*size_px_car[1], size_px_car[2]), left_margin = 5mm, bottom_margin = 5mm)
savefig(p_end, fig_dir*"torus_escape_19_10_2026.png")
display("image/png", p_end)   # PNG only: IJulia would also store an SVG with every point

**Below $g_c$: is the large chaotic set also an attractor?** Six states taken on the chaotic attractor at 77.26 MHz are run at $g_2 < g_c$ for up to $200\,000\,T_a$
($\approx 15\,000/\kappa_b$). The trajectory has landed on the torus once $|a(2nT_a)|$ stays $4000\,T_a$ in the torus band.

In [ ]:
starts_ch = strobe(u_248, 77.26; n_tr = 4000, n_rec = 6000, step = 1000)[1:6]
g_below = [77.236, 77.240, 77.244, 77.246]
life = fill(Inf, length(g_below), length(starts_ch))
@threads for k in 1:length(g_below)*length(starts_ch)
    i, j = divrem(k - 1, length(starts_ch)) .+ 1
    A = section(strobe(starts_ch[j], g_below[i]; n_rec = 200000, step = 20))
    inb = [band[1] < abs(x) < band[2] for x in A]
    n = 200                                   # 200 samples = 4000 T_a
    l = findfirst(m -> all(inb[m:m+n-1]), 1:length(inb)-n+1)
    isnothing(l) || (life[i, j] = 20(l - 1))
end
for i in eachindex(g_below)
    println(g_below[i], " MHz: lifetimes of the chaotic transient (T_a) ", life[i, :], "   median ", median(life[i, :]))
end

## 5. Other cat sizes

Same protocol at $\varepsilon_d/g_2 = 6, 8, 10.12, 13$. The satellite is continued in $\varepsilon_d/g_2$ at 72 MHz from the reference seed, then followed up in $g_2$
(steps of 10 kHz around the Hill line, $3000\,T_a$ between steps). Classification: period from the samples at $T_a$, then $\lambda_1$ over $2000\,T_a$ on the aperiodic points
("aperiodic, regular" if $\lambda_1 < 0.1\,\kappa_b$: a torus, or a periodic orbit longer than 200 iterates or not yet settled; chaotic otherwise). Hill line: `g2_list_12_10_2026_Hill_unstable.json`, first $\varepsilon_p$ of its grid with $|\mu| > 1$,
so an upper bound within one cell ($\approx 0.37$ MHz).

In [ ]:
hill_g = Float64.(JSON.parsefile(out_dir*"g2_list_12_10_2026_Hill_unstable.json"))   # already g2/2π in MHz
hill_r = Float64.(JSON.parsefile(out_dir*"eps_d_ratio_list_12_10_2026_Hill_unstable.json"))
hill_at(r) = (k = findfirst(>=(r), hill_r); hill_r[k] == r ? hill_g[k] : hill_g[k-1] + (hill_g[k] - hill_g[k-1])*(r - hill_r[k-1])/(hill_r[k] - hill_r[k-1]))

ratios = [6.0, 8.0, 10.12, 13.0]
u_72 = strobe(u_seed, 72.0; n_rec = 4000)[end]
sweep = Vector{Any}(undef, length(ratios))
@threads for m in eachindex(ratios)
    r = ratios[m]; gH = hill_at(r)
    u = u_72
    for rr in range(ratio_ref, r, length = 40)
        u = strobe(u, 72.0; r = rr, n_rec = 600)[end]
    end
    gs = vcat(collect(72.0:0.25:gH - 1.5), collect(gH - 1.5:0.01:gH + 1.5))
    A_r = Vector{Vector{ComplexF64}}(undef, length(gs)); cls = zeros(Int, length(gs)); λs = fill(NaN, length(gs))
    for (i, g) in enumerate(gs)
        U = strobe(u, g; r = r, n_tr = 3000, n_rec = 800, step = 1); u = U[end]
        A_r[i] = section(U)
        p = period_of(A_r[i])
        if p in (1, 2)
            cls[i] = p
        else
            λs[i] = λ_max(u, g; r = r, N = 2000)
            cls[i] = λs[i] < 0.1 ? 3 : 4      # 3 quasi-periodic, 4 chaotic
        end
    end
    sweep[m] = (r = r, gH = gH, g = gs, A = A_r, cls = cls, λ = λs)
end
println("first g2/2π (MHz) of each class along the continuation; Hill NS = upper edge of its grid cell")
println(rpad("ε_d/g2", 9), rpad("period 2", 11), rpad("torus", 11), rpad("Hill NS", 11), rpad("chaos", 11), "torus window")
for s in sweep
    f(c) = (k = findfirst(==(c), s.cls); isnothing(k) ? NaN : round(s.g[k], digits=2))
    println(rpad(s.r, 9), rpad(f(2), 11), rpad(f(3), 11), rpad(round(s.gH, digits=2), 11), rpad(f(4), 11), round(f(4) - f(3), digits=2))
end

In [ ]:
cls_col = Perso_palette_hex[[2, 3, 6, 5]]; cls_name = [L"\mathrm{period}\ T_a", L"\mathrm{period}\ 2T_a", "aperiodic, regular", "chaotic"]
plots_r = []
for s in sweep
    w = findall(g -> g >= s.gH - 1.5, s.g)
    p = plot(title = L"\varepsilon_d/2g_2 = "*string(s.r/2), grid = false, legend = false)
    for c in 1:4
        ii = filter(i -> s.cls[i] == c, w)
        isempty(ii) && continue
        scatter!(p, vcat([fill(s.g[i], length(s.A[i])) for i in ii]...), vcat([real.(s.A[i]) for i in ii]...), ms = 0.6, msw = 0, color = cls_col[c], label = "")
    end
    vline!(p, [s.gH], color = :black, ls = :dash, lw = 1, label = "")
    push!(plots_r, p)
end
for c in 1:4; scatter!(plots_r[1], [NaN], [NaN], color = cls_col[c], msw = 0, markershape = :square, label = cls_name[c]); end
plot!(plots_r[1], legend = :bottomleft)
p_ratios = plot(plots_r..., layout = (2, 2), size = (3*size_px_car[1], 2.4*size_px_car[2]), xlabel = L"g_2/2\pi"*" (MHz)", ylabel = L"\mathrm{Re}\,a(nT_a)",
                left_margin = 5mm, bottom_margin = 3mm)
savefig(p_ratios, fig_dir*"torus_window_cat_sizes_19_10_2026.png")
display("image/png", p_ratios)   # PNG only: IJulia would also store an SVG with every point

In [ ]:
open(out_dir*"torus_breakdown_19_10_2026.json", "w") do io
    JSON.print(io, Dict(
        "description" => "Continuation from the satellite at eps_d = 10.12 g2 (and other ratios): torus between the Neimark-Sacker point and chaos. From torus_breakdown.ipynb",
        "kappa_b_code" => κ_b, "T_a" => T_a, "g2_definition" => "E_J sin(eps_p) phi_a^2 phi_b",
        "g_coarse_MHz" => g_coarse, "lambda1_coarse_over_kappa" => λ_coarse, "period_coarse_Ta" => per_coarse,
        "g_zoom_MHz" => g_zoom, "lambda_zoom_over_kappa" => [λ_zoom[i, :] for i in eachindex(g_zoom)], "rotation_zoom" => rot_zoom,
        "abs_a_min_zoom" => amin_zoom, "abs_a_max_zoom" => amax_zoom,
        "g_NS_hill" => g_NS_hill, "g_PD_hill" => g_PD_hill, "theta_NS_hill" => θ_NS_hill, "g_NS_sim" => g_NS_sim, "g_TD_sim" => g_TD_sim, "g_c" => g_c,
        "lambda_conv" => Dict("g" => g_conv, "N_Ta" => N_conv, "lambda" => [λ_conv[i, :] for i in eachindex(g_conv)]),
        "escape" => Dict("delta_MHz" => δ_list, "tau_Ta" => [τ_esc[i, :] for i in eachindex(δ_list)], "gamma_fit" => γ_fit),
        "chaos_lifetime" => Dict("g" => g_below, "tau_Ta" => [life[i, :] for i in eachindex(g_below)]),
        "ratios" => [Dict("ratio" => s.r, "g_hill" => s.gH, "g" => s.g, "class" => s.cls) for s in sweep]))
end

## Conclusion

Along $\varepsilon_d = 10.12\,g_2$ ($\varepsilon_d/2g_2 = 5.06$), following the satellite:

| $g_2/2\pi$ (MHz) | attractor | evidence |
|---|---|---|
| < 75.25 | satellite, period $T_a$ | one point at $T_a$ |
| 75.25 | period doubling | Hill 75.25 MHz; two points at $T_a$ |
| 76.70 | Neimark–Sacker → torus $T^2$ | Hill 76.69 MHz. Section at $2T_a$: one smooth closed curve. $\lambda_1 \to 0$ as $1/T$ ($\lambda_1\kappa_b T \approx 3$ for $\kappa_b T$ = 37 to 2400), $\lambda_2 < 0$. Rotation number 0.106 at the onset, Hill $\arg\mu/2\pi$ = 0.107 |
| ≈ 77.0 | torus doubling | $\lambda_2$ reaches 0 again; the curve becomes one curve winding twice (even and odd iterates fill both loops) |
| 77.249 | jump to a large chaotic attractor | $\lambda_1 \approx 0.6$–$0.7\,\kappa_b$, and $\lvert a\rvert$ spans 4.2–5.9 instead of 5.06–5.29 on the torus |

**Is it a torus breakdown?** Not in the usual sense (a smooth torus becomes wrinkled, often after phase locking, and a small chaotic attractor grows out of it).
- Just below $g_c$ the doubled torus is still smooth at the resolution of the sections and transversally stable: $\lambda/\kappa_b = (0.011, -0.067, -0.082, -1.86)$ at 77.2488 MHz. So it is not destroyed by a local bifurcation (a further doubling, a saddle-node of tori or a secondary Neimark–Sacker would need $\lambda_2 \to 0$).
- No locked orbit (period ≤ 200 iterates) was found along the window. The rotation number increases continuously from 0.106 to 0.113.
- The chaotic set exists before the torus disappears, as a chaotic saddle: started on it at 77.236–77.246 MHz, every trajectory ends on the torus after $10^3$–$10^5\,T_a$ (median lifetime of order $10^4\,T_a$, growing towards $g_c$). Above $g_c$, trajectories leave the ghost of the torus after a time $\tau \propto (g_2 - g_c)^{-\gamma}$, $\gamma \approx 0.37$ (670 $T_a$ at 0.3 kHz, 73 $T_a$ at 100 kHz).

So the route is **period doubling → Neimark–Sacker → torus → torus doubling → abrupt (crisis-type) transition to chaos**. This is the truncated torus-doubling scenario
(Kaneko 1983; Arnéodo, Coullet and Spiegel 1983), ended by a global event rather than by a gradual loss of smoothness. Which saddle the torus collides with is not identified here;
that would need the saddle orbits on the basin boundary between the torus and the chaotic set.

**Other cat sizes.** At $\varepsilon_d/g_2 = 6, 8, 13$ the same sequence appears, with a quasi-periodic window 0.55–0.7 MHz wide that starts within the Hill grid cell and ends abruptly.
The finer structure (torus doubling) was only checked at 10.12. At $\varepsilon_d/g_2 = 13$ the chaos is interrupted at 76.2–76.55 MHz by a periodic window
(about 20 branches in the diagram, i.e. a long-period orbit, counted as "aperiodic, regular" by the classifier), then resumes.

**Consequences for the article (v2).**
- App. "Quasi-periodic orbits": the window is 76.70–77.25 MHz (0.55 MHz). It lies between the grid points 76.5 and 77.6 MHz of the maps, so they never sample it at this cat size. The statement that the onset of chaos follows the Neimark–Sacker line within one grid cell holds.
- App. "Confined chaos": "the torus, and then the chaotic set, are born from the period-doubled satellite and stay close to it" should be nuanced. The torus is local (width ≈ 0.25 in $\lvert a\rvert$), but the chaotic attractor appears abruptly and is about seven times wider ($\lvert a\rvert$ 4.2–5.9). It still stays far from the cats ($\lvert a\rvert \approx 2.2$).
- The answer to open question 4 is: period doubling, then Neimark–Sacker, then a torus that doubles once and is destroyed abruptly. It is not a gradual torus breakdown.

All times here are in the code convention ($\kappa_b = \pi/10.4$). In the article's convention, $\kappa_b T$ values double and $\lambda/\kappa_b$ values halve.